In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog.silver;

In [0]:
%sql
SELECT * FROM catalog.bronze.patients_raw;

In [0]:
from pyspark.sql.functions import sha2, col, concat_ws, monotonically_increasing_id, current_timestamp, when, upper, coalesce, try_to_date
from delta.tables import DeltaTable


In [0]:
bronze_table = 'catalog.bronze.patients_raw'
silver_table = 'catalog.silver.dim_patients'
checkpoint_path = "abfss://data@datastoragea.dfs.core.windows.net/silver/dim_patients/checkpoint/"

df_patients_bronze = spark.readStream.table(bronze_table)

df_patients_clean = (
    df_patients_bronze
        .drop(
            "philhealth_no",
            "contact_number",
            "blood_type",
            "_rescued_data",
            "_source_file",
            "_ingest_timestamp",
            "ingestion_date"
        )
        .withColumn(
            "gender",
            when(upper(col("gender")).isin("MALE", "M"), "M")
            .when(upper(col("gender")).isin("FEMALE", "F"), "F")
            .otherwise(None)
            )
        .withColumn(
            "dob", coalesce(
                try_to_date(col("dob"), "yyyy-MM-dd"),
                try_to_date(col("dob"), "MM/dd/yyyy"),
                try_to_date(col("dob"), "dd-MM-yyyy"),
                try_to_date(col("dob"), "MMMM dd yyyy"),   # handles "November 27 1971"
            )
        )
        .withColumn("patient_name_hash", sha2(concat_ws('|', col("first_name"), col("last_name")), 256))
        .withColumn("load_timestamp", current_timestamp())
)

def merge_dim_patients(batch_df, batch_id):
    batch_df = batch_df.dropDuplicates(["patient_id"])

    if not spark.catalog.tableExists(silver_table):
        # First run — create the table
        (batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table))  
        return
    
    # Load Delta table by name and upsert into Silver
    dim_patients = DeltaTable.forName(spark, silver_table)

    (dim_patients.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.patient_id = s.patient_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    

In [0]:
(   
    df_patients_clean.writeStream
        .foreachBatch(merge_dim_patients)  # for every batch, merge_dim_patients is run
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)

In [0]:
%sql
SELECT * FROM catalog.silver.dim_patients;

In [0]:
# Data quality check
from pyspark.sql.functions import col, count, min, max, year, current_date, datediff

print("DIM PATIENTS")
df = spark.read.table("catalog.silver.dim_patients")
total = df.count()
print(f"Total rows: {total} (expected: 500)")

dup_id = total - df.select("patient_id").distinct().count()
print(f"Duplicate patient_ids: {'OK' if dup_id == 0 else dup_id}")

for c in ["patient_id", "gender", "dob", "assigned_branch"]:
    n = df.filter(col(c).isNull()).count()
    print(f"  {'OK' if n == 0 else 'CHECK'} {c}: {n} nulls")

# Gender should only be M or F after standardization
invalid_gender = df.filter(~col("gender").isin("M", "F")).count()
print(f"Invalid gender values: {'OK' if invalid_gender == 0 else invalid_gender}")

print("Gender distribution:")
df.groupBy("gender").count().show()

# DOB sanity — no future dates, no unrealistic ages
df = df.withColumn("age", (datediff(current_date(), col("dob")) / 365).cast("int"))
df.select(
    min("age").alias("min_age"),
    max("age").alias("max_age")
).show()

# Assigned branch should all be valid: H001-H010
invalid_branch = df.filter(
    ~col("assigned_branch").isin([f"H{i:03d}" for i in range(1,11)])
).count()
print(f"Invalid assigned_branch: {'OK' if invalid_branch == 0 else f'CHECK — {invalid_branch} rows'}")